In [1]:
import boto3
import json
from pydantic import BaseModel, Field
from typing import List, Optional
from datetime import datetime
import os
from dotenv import load_dotenv
import re
import uuid
import mlflow
import dagshub

# Load env file
load_dotenv()

os.environ["MLFLOW_TRACKING_USERNAME"] = os.getenv("DAGSHUB_USERNAME")
os.environ["MLFLOW_TRACKING_PASSWORD"] = os.getenv("DAGSHUB_TOKEN")

# Tracking server
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))
mlflow.set_experiment("inframind_experiment")

AWS_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")

bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name="ap-south-1",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY
)

x:\nasim_xhqpjmy\Code\GenAI\InfraMind\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Llama 3 Pricing on AWS Bedrock (On-Demand per 1k tokens)
PRICING = {
    "meta.llama3-8b-instruct-v1:0": {"input": 0.0003, "output": 0.0006},
    "meta.llama3-70b-instruct-v1:0": {"input": 0.00265, "output": 0.0035}
}

In [3]:
class RCAOutput(BaseModel):
    incident_id: str = Field(description="Unique ID for the incident")
    severity: str = Field(description="Critical, High, Medium, or Low")
    summary: str = Field(description="One-sentence summary of what happened")
    root_cause: str = Field(description="The primary technical reason for the failure")
    immediate_fix: str = Field(description="What to do right now to restore service")
    confidence_score: float = Field(description="AI's confidence (0.0 to 1.0)")
    model_used: str = Field(description="Llama-3-8B or Llama-3-70B")

json_schema = RCAOutput.model_json_schema()

In [4]:
def track_usage(response_body, model_id):
    """Calculates cost and logs tokens to MLflow"""
    i_tokens = response_body.get("prompt_token_count", 0)
    o_tokens = response_body.get("generation_token_count", 0)
    
    cost = (i_tokens * PRICING[model_id]["input"]) + (o_tokens * PRICING[model_id]["output"])
    
    # Log to MLflow (Assuming a run is active)
    mlflow.log_metric(f"tokens_in", i_tokens)
    mlflow.log_metric(f"tokens_out", o_tokens)
    mlflow.log_metric(f"cost_usd", cost)
    
    return cost

In [5]:
def extract_json(text: str) -> str:
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("No JSON object found in model output")
    return text[start:end + 1]

In [6]:
def generate_infra_rca(raw_log: str, context: str = "",feedback: str = "") -> RCAOutput:

    # 0. Generate a unique incident ID for tracking
    incident_id = str(uuid.uuid4())

    # 1. Simple Routing Logic
    # If the log is massive, we escalate to the 70B model
    selected_model = "meta.llama3-70b-instruct-v1:0" if len(raw_log) > 2000 else "meta.llama3-8b-instruct-v1:0"
    
    # 2. Construct the Prompt

    # Inject feedback if this is a retry
    retry_context = f"\n[CRITICAL FEEDBACK FROM PREVIOUS ATTEMPT]:\n{feedback}\n Please fix the issues mentioned above." if feedback else ""
    
    prompt = f"""
    <|begin_of_text|>

    <|start_header_id|>system<|end_header_id|>
    You are a Senior Site Reliability Engineer (SRE).

    Your task is to analyze infrastructure incident logs and produce a Root Cause Analysis (RCA).

    Rules:
    - Return ONLY valid JSON.
    - Do NOT include explanations or markdown.
    - Follow the JSON schema exactly.
    - Always include the provided incident_id.
    - The incident_id for this task is: {incident_id}

    JSON schema:
    {json.dumps(json_schema)}

    {retry_context}
    <|eot_id|>

    <|start_header_id|>user<|end_header_id|>

    Incident Context:
    {context}

    Incident Log:
    {raw_log}

    Return the RCA JSON.
    <|eot_id|>

    <|start_header_id|>assistant<|end_header_id|>
    """

    # 3. Call AWS Bedrock
    response = bedrock_runtime.invoke_model(
        modelId=selected_model,
        contentType="application/json",
        accept="application/json",
        body=json.dumps({
            "prompt": prompt,
            "max_gen_len": 1024,
            "temperature": 0.1,
            "top_p": 0.9,
            "stop": ["\n\n"]
        })
    )
    
    # 4. Parse and Validate
    result = json.loads(response["body"].read())
    track_usage(result, selected_model)         #log cost and tokens to MLflow
    generation = result["generation"]

    # Extract JSON safely
    json_text = extract_json(generation)

    try:
        data = json.loads(json_text)
        rca = RCAOutput.model_validate(data)
    except Exception as e:
        raise ValueError(f"Model returned invalid JSON: {json_text}") from e

    # Force correct metadata
    rca.incident_id = incident_id
    model_label = "Llama-3-70B" if "70b" in selected_model else "Llama-3-8B"
    rca.model_used = model_label

    return rca

In [7]:
test_log = "ERROR: [Kubelet] Failed to pull image 'nginx:latest': RPC error: code = Unknown desc = Error response from daemon: Get https://registry-1.docker.io/v2/: net/http: TLS handshake timeout"

try:
    analysis = generate_infra_rca(test_log, context="VPC Security Groups allow outbound 443.")
    print("\n--- RCA REPORT ---")
    print("Incident ID:", analysis.incident_id)
    print("Severity:", analysis.severity)
    print("Summary:", analysis.summary)
    print("Root Cause:", analysis.root_cause)
    print("Immediate Fix:", analysis.immediate_fix)
    print("Confidence:", analysis.confidence_score)
    print("Model:", analysis.model_used)
except Exception as e:
    print(f"Mission Failed: {e}")


--- RCA REPORT ---
Incident ID: 8e4cf6a4-82fb-4efc-8a35-c36353adbf16
Severity: High
Summary: Kubelet failed to pull image due to TLS handshake timeout
Root Cause: VPC Security Groups not allowing outbound traffic on port 443
Immediate Fix: Allow outbound traffic on port 443 in VPC Security Groups
Confidence: 0.9
Model: Llama-3-8B


In [8]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_aws import BedrockEmbeddings

# Load documents
loader = DirectoryLoader("runbook", glob="*.md", loader_cls=TextLoader)
docs = loader.load()

print("Docs:", len(docs))

headers_to_split_on = [
    ("#", "Header1"),
    ("##", "Header2"),
    ("###", "Header3"),
]

splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

splits = []

for doc in docs:
    splits.extend(splitter.split_text(doc.page_content))

print("Splits:", len(splits))

embeddings = BedrockEmbeddings(
    client=bedrock_runtime,
    model_id="amazon.titan-embed-text-v2:0"
)

vector_db = FAISS.from_documents(
    splits,
    embeddings
)

def get_context(query):
    results = vector_db.similarity_search(query, k=4)
    return "\n---\n".join([doc.page_content for doc in results])

Docs: 3
Splits: 41


In [9]:
from langchain_aws import ChatBedrock
from deepeval.models.base_model import DeepEvalBaseLLM
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

class BedrockJudge(DeepEvalBaseLLM):
    """Thin wrapper so DeepEval can use Bedrock's 70B as its LLM judge."""

    def __init__(self):
        self.chat_model = ChatBedrock(
            client=bedrock_runtime,
            model_id="meta.llama3-70b-instruct-v1:0",
            model_kwargs={"temperature": 0.0},
        )

    def load_model(self):
        return self.chat_model

    def generate(self, prompt: str) -> str:
        model = self.load_model()
        return model.invoke(prompt).content

    async def a_generate(self, prompt: str) -> str:
        model = self.load_model()
        res = await model.ainvoke(prompt)
        return res.content

    def get_model_name(self) -> str:
        return "Bedrock Llama-3-70B Judge"

# Instantiate once — reused across all evaluations
bedrock_judge = BedrockJudge()


In [10]:
def run_deepeval(log, context, output_json):
    try:
        test_case = LLMTestCase(
            input=log,
            actual_output=output_json.immediate_fix,
            retrieval_context=[context]
        )

        # Pass bedrock_judge so DeepEval never tries to call OpenAI
        faith_metric = FaithfulnessMetric(threshold=0.7, model=bedrock_judge)
        rel_metric = AnswerRelevancyMetric(threshold=0.7, model=bedrock_judge)

        faith_metric.measure(test_case)
        rel_metric.measure(test_case)

        return faith_metric.score, rel_metric.score

    except Exception as e:
        print(f"DeepEval evaluation failed: {e}")
        return 0.0, 0.0

In [11]:
def senior_sre_critique(rca_report: RCAOutput, runbook_context: str):
    """The 70B model acts as a Senior SRE to find flaws in the 8B's output."""
    
    critique_prompt = f"""
    <|begin_of_text|>
    <|start_header_id|>system<|end_header_id|>
    You are a Staff SRE reviewing an RCA report for technical accuracy.
    Score the RCA from 0-10.
    Return output strictly as:

    SCORE: [X] | NOTE: explanation
    <|eot_id|>

    <|start_header_id|>user<|end_header_id|>
    Runbook context:
    {runbook_context}

    RCA Report:
    {rca_report.model_dump_json()}
    <|eot_id|>

    <|start_header_id|>assistant<|end_header_id|>
    """

    response = bedrock_runtime.invoke_model(
        modelId="meta.llama3-70b-instruct-v1:0",
        contentType="application/json",
        accept="application/json",
        body=json.dumps({
            "prompt": critique_prompt,
            "max_gen_len": 512,
            "temperature": 0.1
        })
    )
    
    result = json.loads(response['body'].read())

    print("\n--- RAW CRITIQUE RESPONSE ---")
    print(result)

    critique_text = result.get("generation", "")
    return critique_text

In [12]:
def run_autonomous_workflow(log_text, max_retries=2):
    if mlflow.active_run():
        mlflow.end_run()

    with mlflow.start_run(run_name=f"Incident_{datetime.now().strftime('%H%M%S')}"):
        mlflow.log_param("incident_log", log_text)
        
        knowledge = get_context(f"incident: {log_text} troubleshooting runbook")
        mlflow.log_text(knowledge, "retrieved_context.txt")
        mlflow.log_param("retrieved_context_length", len(knowledge))
        current_feedback = ""
        
        for attempt in range(max_retries + 1):
            print(f"\n🚀 Execution Attempt {attempt + 1}...")
            
            # Generate RCA
            rca = generate_infra_rca(log_text, context=knowledge, feedback=current_feedback)
        
            mlflow.log_metric(f"confidence_score_attempt_{attempt+1}", rca.confidence_score)
            mlflow.log_metric("rca_length", len(rca.summary))
            
            mlflow.log_dict({
                "incident_id": rca.incident_id,
                "severity": rca.severity,
                "summary": rca.summary,
                "root_cause": rca.root_cause,
                "immediate_fix": rca.immediate_fix
            }, f"rca_output_attempt_{attempt+1}.json")
            
            # Senior SRE critique
            critique = senior_sre_critique(rca, knowledge)
            print(f"Critique Received: {critique.strip()}")
            mlflow.log_text(critique, f"critique_attempt_{attempt+1}.txt")

            faith_score, rel_score = run_deepeval(
                        log_text,
                        knowledge,
                        rca)
            
            mlflow.log_metric(f"faithfulness_score_attempt_{attempt+1}", faith_score)
            mlflow.log_metric(f"relevancy_score_attempt_{attempt+1}", rel_score)

            # Parse Score
            match = re.search(r"score\s*:\s*\[?(\d+)\]?", critique, re.IGNORECASE)
            score = float(match.group(1)) / 10 if match else 0.0
            
            mlflow.log_metric(f"attempt_{attempt+1}_score", score)
            
            # 4. Check quality threshold
            if score >= 0.8:
                print(f"✅ Success! Quality score {score} meets threshold.")
                mlflow.log_dict(rca.model_dump(), "final_rca_output.json")
                mlflow.log_text(critique, "final_senior_sre_review.txt")
                return rca
            else:
                print(f"⚠️ Score too low ({score}). Triggering self-correction loop...")
                current_feedback = critique # Pass the bad grade back into the AI
                
        print("🚨 Max retries reached. Returning best effort.")
        return rca

In [13]:
if __name__ == "__main__":
    new_incident_log = "ERROR 500: Database connection refused while connecting to host rds.cluster-inframind.internal"
    
    try:
        final_rca = run_autonomous_workflow(new_incident_log)
        
        print("\n--- FINAL ANALYTICS REPORT ---")
        print("Incident ID:", final_rca.incident_id)
        print("Severity:", final_rca.severity)
        print("Summary:", final_rca.summary)
        print("Root Cause:", final_rca.root_cause)
        print("Immediate Fix:", final_rca.immediate_fix)
        print("Confidence:", final_rca.confidence_score)
        print("Model:", final_rca.model_used)
        
    except Exception as e:
        print(f"System Failure: {e}")

🏃 View run unique-stork-840 at: https://dagshub.com/nasim-raj-laskar/InfraMind.mlflow/#/experiments/0/runs/20f8545317a141e3954b1271e3bbbb7a
🧪 View experiment at: https://dagshub.com/nasim-raj-laskar/InfraMind.mlflow/#/experiments/0

🚀 Execution Attempt 1...

--- RAW CRITIQUE RESPONSE ---
{'generation': '\n\n    SCORE: 8 | NOTE: The RCA report is mostly accurate, but it lacks technical details and supporting evidence to increase confidence in the root cause analysis. The report correctly identifies the symptoms and provides relevant immediate actions, but the root cause analysis could be more comprehensive, considering other potential factors contributing to the issue. Additionally, the confidence score of 0.8 seems subjective and could be justified with more technical explanations.', 'prompt_token_count': 337, 'generation_token_count': 86, 'stop_reason': 'stop'}
Critique Received: SCORE: 8 | NOTE: The RCA report is mostly accurate, but it lacks technical details and supporting evidence

x:\nasim_xhqpjmy\Code\GenAI\InfraMind\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

✅ Success! Quality score 0.8 meets threshold.
🏃 View run Incident_170733 at: https://dagshub.com/nasim-raj-laskar/InfraMind.mlflow/#/experiments/0/runs/e5c2532bb6454842aec2ac69fecc3d0a
🧪 View experiment at: https://dagshub.com/nasim-raj-laskar/InfraMind.mlflow/#/experiments/0

--- FINAL ANALYTICS REPORT ---
Incident ID: 5033a2e9-c398-435d-b031-b961378c6d7e
Severity: High
Summary: Database connection refused due to connection pool misconfiguration
Root Cause: Connection pool misconfiguration
Immediate Fix: Check database server status, verify connection limits, and kill long-running queries
Confidence: 0.8
Model: Llama-3-8B
